# 04 — Ingest Payments into Bronze

## Purpose

Incrementally ingest deterministic payment-attempt history generated by the Payment Gateway into a governed Delta Bronze table.

The notebook preserves successful, failed, pending, and retry attempts; applies the payment data contract; captures operational lineage; and reconciles successful settlements with paid invoice balances.

## Business Grain

One row represents one payment attempt against an invoice.

A single invoice can have multiple attempts when the first payment fails and the Payment Gateway performs a retry.

## Design

- Read payment-attempt JSON files from the canonical Payment Gateway path.
- Apply an explicit payment event schema.
- Validate identifiers, invoice ownership, statuses, retry sequences, dates, and settlements.
- Reconcile successful payments with paid invoice balances.
- Ingest incrementally with Databricks Auto Loader.
- Add technical lineage and deterministic record hashes.
- Store immutable payment events in a Delta Bronze table.
- Reconcile Bronze content exactly with the source.
- Verify checkpoint-based idempotency.

## Source

- `/Volumes/workspace/revenue_leakage_bronze/landing/payment_gateway/payments`

## Target

- `workspace.revenue_leakage_bronze.payment_events`

## Expected First-Load Volume

- 30,232 payment attempts
- 22,095 successful attempts
- 7,940 failed attempts
- 197 pending attempts
- 30,232 total Bronze events

In [0]:
from decimal import Decimal

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType,
    IntegerType,
    DecimalType,
)

SOURCE_SYSTEM = "Payment Gateway"
SOURCE_ENTITY = "payments"
SOURCE_FORMAT = "json"

EXPECTED_PAYMENT_EVENT_COUNT = 30_232

EXPECTED_OPERATION_COUNTS = {
    "INSERT": 30_232,
}

EXPECTED_PAYMENT_STATUS_COUNTS = {
    "Failed": 7_940,
    "Pending": 197,
    "Succeeded": 22_095,
}

EXPECTED_STATUS_ATTEMPT_COUNTS = {
    ("Failed", 1): 6_982,
    ("Failed", 2): 958,
    ("Pending", 1): 197,
    ("Succeeded", 1): 18_787,
    ("Succeeded", 2): 3_308,
}

EXPECTED_TRANSACTION_TOTAL = Decimal(
    "5285980.29"
)

EXPECTED_SETTLEMENT_TOTAL = Decimal(
    "3843928.50"
)

LANDING_PATH = (
    "/Volumes/workspace/"
    "revenue_leakage_bronze/landing"
)

PAYMENTS_SOURCE_PATH = (
    f"{LANDING_PATH}/"
    "payment_gateway/payments"
)

PAYMENTS_INITIAL_PATH = (
    f"{PAYMENTS_SOURCE_PATH}/initial_load"
)

INVOICES_REFERENCE_PATH = (
    f"{LANDING_PATH}/"
    "billing_system/invoices/initial_load"
)

PAYMENTS_SCHEMA_PATH = (
    f"{LANDING_PATH}/_schemas/"
    "bronze/payment_gateway/payments"
)

PAYMENTS_CHECKPOINT_PATH = (
    f"{LANDING_PATH}/_checkpoints/"
    "bronze/payment_gateway/payments"
)

PAYMENTS_BRONZE_TABLE = (
    "workspace.revenue_leakage_bronze."
    "payment_events"
)

PAYMENT_EVENT_SCHEMA = StructType([
    StructField(
        "payment_id",
        StringType(),
        False,
    ),
    StructField(
        "provider_transaction_id",
        StringType(),
        False,
    ),
    StructField(
        "invoice_id",
        StringType(),
        False,
    ),
    StructField(
        "subscription_id",
        StringType(),
        False,
    ),
    StructField(
        "customer_id",
        StringType(),
        False,
    ),
    StructField(
        "transaction_amount",
        DecimalType(12, 2),
        False,
    ),
    StructField(
        "settled_amount",
        DecimalType(12, 2),
        False,
    ),
    StructField(
        "currency",
        StringType(),
        False,
    ),
    StructField(
        "payment_status",
        StringType(),
        False,
    ),
    StructField(
        "attempt_number",
        IntegerType(),
        False,
    ),
    StructField(
        "payment_method",
        StringType(),
        False,
    ),
    StructField(
        "payment_provider",
        StringType(),
        False,
    ),
    StructField(
        "failure_reason",
        StringType(),
        True,
    ),
    StructField(
        "payment_date",
        DateType(),
        False,
    ),
    StructField(
        "settlement_date",
        DateType(),
        True,
    ),
    StructField(
        "operation",
        StringType(),
        False,
    ),
    StructField(
        "event_timestamp",
        TimestampType(),
        False,
    ),
    StructField(
        "snapshot_date",
        DateType(),
        False,
    ),
])

PAYMENT_SOURCE_COLUMNS = (
    PAYMENT_EVENT_SCHEMA.fieldNames()
)

OPTIONAL_PAYMENT_COLUMNS = {
    "failure_reason",
    "settlement_date",
}

BRONZE_METADATA_COLUMNS = [
    "_source_system",
    "_source_entity",
    "_source_file_path",
    "_source_file_name",
    "_source_file_size",
    "_source_file_modification_time",
    "_ingested_at",
    "_ingestion_date",
    "_record_hash",
    "_rescued_data",
]

PAYMENT_BRONZE_COLUMNS = (
    PAYMENT_SOURCE_COLUMNS
    + BRONZE_METADATA_COLUMNS
)

## 2. Load and Validate the Payment Source

Load the payment-attempt history and invoice reference dataset, then validate expected volumes, identifiers, ownership, payment states, retry sequences, dates, transaction amounts, and settlement reconciliation before Bronze ingestion.

In [0]:
INVOICE_PAYMENT_REFERENCE_SCHEMA = StructType([
    StructField(
        "invoice_id",
        StringType(),
        False,
    ),
    StructField(
        "subscription_id",
        StringType(),
        False,
    ),
    StructField(
        "customer_id",
        StringType(),
        False,
    ),
    StructField(
        "invoice_date",
        DateType(),
        False,
    ),
    StructField(
        "invoice_total_amount",
        DecimalType(12, 2),
        False,
    ),
    StructField(
        "invoice_status",
        StringType(),
        False,
    ),
])

payments_source_df = (
    spark.read
    .format(SOURCE_FORMAT)
    .schema(PAYMENT_EVENT_SCHEMA)
    .load(PAYMENTS_INITIAL_PATH)
    .withColumn(
        "_landing_batch",
        F.lit("initial_load"),
    )
)

invoice_reference_df = (
    spark.read
    .format("json")
    .schema(INVOICE_PAYMENT_REFERENCE_SCHEMA)
    .load(INVOICES_REFERENCE_PATH)
)

payment_validation_df = (
    payments_source_df.alias("payment")
    .join(
        invoice_reference_df.alias("invoice"),
        on=(
            F.col("payment.invoice_id")
            == F.col("invoice.invoice_id")
        ),
        how="left",
    )
    .select(
        "payment.*",
        F.col(
            "invoice.invoice_id"
        ).alias(
            "_reference_invoice_id"
        ),
        F.col(
            "invoice.subscription_id"
        ).alias(
            "_invoice_subscription_id"
        ),
        F.col(
            "invoice.customer_id"
        ).alias(
            "_invoice_customer_id"
        ),
        F.col(
            "invoice.invoice_date"
        ).alias(
            "_invoice_date"
        ),
        F.col(
            "invoice.invoice_total_amount"
        ).alias(
            "_invoice_total_amount"
        ),
        F.col(
            "invoice.invoice_status"
        ).alias(
            "_invoice_status"
        ),
    )
)

required_payment_columns = [
    column_name
    for column_name in PAYMENT_SOURCE_COLUMNS
    if column_name not in OPTIONAL_PAYMENT_COLUMNS
]

required_field_is_missing = None

for column_name in required_payment_columns:
    missing_condition = (
        F.col(column_name).isNull()
        | (
            F.trim(
                F.col(column_name).cast("string")
            )
            == F.lit("")
        )
    )

    required_field_is_missing = (
        missing_condition
        if required_field_is_missing is None
        else required_field_is_missing
        | missing_condition
    )

invalid_domain_condition = (
    ~F.col("payment_status").isin(
        "Failed",
        "Pending",
        "Succeeded",
    )
    | ~F.col("attempt_number").isin(
        1,
        2,
    )
    | (F.col("currency") != "USD")
    | (F.col("operation") != "INSERT")
    | (F.col("transaction_amount") <= 0)
    | (F.col("settled_amount") < 0)
)

invalid_ownership_condition = (
    F.col("_reference_invoice_id").isNotNull()
    & (
        (
            F.col("subscription_id")
            != F.col("_invoice_subscription_id")
        )
        | (
            F.col("customer_id")
            != F.col("_invoice_customer_id")
        )
    )
)

invalid_transaction_condition = (
    F.col("_invoice_total_amount").isNotNull()
    & (
        F.abs(
            F.col("transaction_amount")
            - F.col("_invoice_total_amount")
        )
        > F.lit(0.01)
    )
)

invalid_payment_date_condition = (
    (
        F.col("_invoice_date").isNotNull()
        & (
            F.col("payment_date")
            < F.col("_invoice_date")
        )
    )
    | (
        F.col("payment_date")
        > F.col("snapshot_date")
    )
    | (
        F.to_date("event_timestamp")
        > F.col("snapshot_date")
    )
    | (
        F.col("settlement_date").isNotNull()
        & (
            F.col("settlement_date")
            < F.col("payment_date")
        )
    )
    | (
        F.col("settlement_date").isNotNull()
        & (
            F.col("settlement_date")
            > F.col("snapshot_date")
        )
    )
)

invalid_settlement_state_condition = (
    (
        (F.col("payment_status") == "Succeeded")
        & (
            F.col("settlement_date").isNull()
            | (
                F.abs(
                    F.col("settled_amount")
                    - F.col("transaction_amount")
                )
                > F.lit(0.01)
            )
        )
    )
    | (
        (F.col("payment_status") != "Succeeded")
        & (
            F.col("settlement_date").isNotNull()
            | (F.col("settled_amount") != 0)
        )
    )
)

invalid_failure_state_condition = (
    (
        (F.col("payment_status") == "Failed")
        & (
            F.col("failure_reason").isNull()
            | (
                F.trim("failure_reason")
                == F.lit("")
            )
        )
    )
    | (
        (F.col("payment_status") != "Failed")
        & F.col("failure_reason").isNotNull()
    )
)

invalid_invoice_status_mapping_condition = (
    (
        (F.col("payment_status") == "Succeeded")
        & (F.col("_invoice_status") != "Paid")
    )
    | (
        (F.col("payment_status") == "Pending")
        & (F.col("_invoice_status") != "Open")
    )
    | (
        (F.col("payment_status") == "Failed")
        & (
            ~F.col("_invoice_status").isin(
                "Paid",
                "Past Due",
            )
        )
    )
)

payment_source_metrics = (
    payment_validation_df
    .agg(
        F.count("*").alias(
            "payment_event_count"
        ),

        F.countDistinct(
            "payment_id"
        ).alias(
            "distinct_payment_id_count"
        ),

        F.countDistinct(
            "provider_transaction_id"
        ).alias(
            "distinct_provider_transaction_count"
        ),

        F.countDistinct(
            F.struct(
                "invoice_id",
                "attempt_number",
            )
        ).alias(
            "distinct_invoice_attempt_count"
        ),

        F.sum(
            F.when(
                required_field_is_missing,
                1,
            ).otherwise(0)
        ).alias(
            "null_required_field_count"
        ),

        F.sum(
            F.when(
                F.col(
                    "_reference_invoice_id"
                ).isNull(),
                1,
            ).otherwise(0)
        ).alias(
            "orphan_invoice_count"
        ),

        F.sum(
            F.when(
                invalid_ownership_condition,
                1,
            ).otherwise(0)
        ).alias(
            "ownership_mismatch_count"
        ),

        F.sum(
            F.when(
                invalid_transaction_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_transaction_count"
        ),

        F.sum(
            F.when(
                invalid_domain_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_domain_count"
        ),

        F.sum(
            F.when(
                invalid_payment_date_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_payment_date_count"
        ),

        F.sum(
            F.when(
                invalid_settlement_state_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_settlement_state_count"
        ),

        F.sum(
            F.when(
                invalid_failure_state_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_failure_state_count"
        ),

        F.sum(
            F.when(
                invalid_invoice_status_mapping_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_invoice_status_mapping_count"
        ),

        F.round(
            F.sum("transaction_amount"),
            2,
        ).alias(
            "transaction_total"
        ),

        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settlement_total"
        ),
    )
    .first()
    .asDict()
)

payment_event_count = int(
    payment_source_metrics[
        "payment_event_count"
    ]
)

distinct_payment_id_count = int(
    payment_source_metrics[
        "distinct_payment_id_count"
    ]
)

distinct_provider_transaction_count = int(
    payment_source_metrics[
        "distinct_provider_transaction_count"
    ]
)

distinct_invoice_attempt_count = int(
    payment_source_metrics[
        "distinct_invoice_attempt_count"
    ]
)

duplicate_payment_count = (
    payment_event_count
    - distinct_payment_id_count
)

duplicate_provider_transaction_count = (
    payment_event_count
    - distinct_provider_transaction_count
)

duplicate_attempt_count = (
    payment_event_count
    - distinct_invoice_attempt_count
)

attempt_sequence_summary_df = (
    payments_source_df
    .groupBy("invoice_id")
    .agg(
        F.min("attempt_number").alias(
            "first_attempt_number"
        ),
        F.max("attempt_number").alias(
            "last_attempt_number"
        ),
        F.countDistinct(
            "attempt_number"
        ).alias(
            "distinct_attempt_count"
        ),
    )
)

invalid_attempt_sequence_count = (
    attempt_sequence_summary_df
    .filter(
        (F.col("first_attempt_number") != 1)
        | (F.col("last_attempt_number") > 2)
        | (
            F.col("distinct_attempt_count")
            != F.col("last_attempt_number")
        )
    )
    .count()
)

first_attempts_df = (
    payments_source_df
    .filter(
        F.col("attempt_number") == 1
    )
    .select(
        F.col("invoice_id").alias(
            "_first_invoice_id"
        ),
        F.col("payment_status").alias(
            "_first_payment_status"
        ),
        F.col("payment_date").alias(
            "_first_payment_date"
        ),
    )
)

second_attempts_validation_df = (
    payments_source_df
    .filter(
        F.col("attempt_number") == 2
    )
    .join(
        first_attempts_df,
        on=(
            F.col("invoice_id")
            == F.col("_first_invoice_id")
        ),
        how="left",
    )
)

invalid_retry_predecessor_count = (
    second_attempts_validation_df
    .filter(
        F.col("_first_invoice_id").isNull()
        | (
            F.col("_first_payment_status")
            != "Failed"
        )
        | (
            F.col("payment_date")
            < F.col("_first_payment_date")
        )
    )
    .count()
)

successful_invoice_ids_df = (
    payments_source_df
    .filter(
        F.col("payment_status") == "Succeeded"
    )
    .select("invoice_id")
    .distinct()
)

failed_invoice_ids_df = (
    payments_source_df
    .filter(
        F.col("payment_status") == "Failed"
    )
    .select("invoice_id")
    .distinct()
)

paid_invoices_without_success_count = (
    invoice_reference_df
    .filter(
        F.col("invoice_status") == "Paid"
    )
    .select("invoice_id")
    .join(
        successful_invoice_ids_df,
        on="invoice_id",
        how="left_anti",
    )
    .count()
)

past_due_invoices_without_failure_count = (
    invoice_reference_df
    .filter(
        F.col("invoice_status") == "Past Due"
    )
    .select("invoice_id")
    .join(
        failed_invoice_ids_df,
        on="invoice_id",
        how="left_anti",
    )
    .count()
)

duplicate_successful_invoice_count = (
    payments_source_df
    .filter(
        F.col("payment_status") == "Succeeded"
    )
    .groupBy("invoice_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

voided_invoice_payment_count = (
    payment_validation_df
    .filter(
        F.col("_invoice_status") == "Voided"
    )
    .count()
)

successful_settlement_by_invoice_df = (
    payments_source_df
    .filter(
        F.col("payment_status") == "Succeeded"
    )
    .groupBy("invoice_id")
    .agg(
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "successful_settlement_amount"
        )
    )
)

paid_settlement_mismatch_count = (
    invoice_reference_df
    .filter(
        F.col("invoice_status") == "Paid"
    )
    .join(
        successful_settlement_by_invoice_df,
        on="invoice_id",
        how="inner",
    )
    .filter(
        F.abs(
            F.col("invoice_total_amount")
            - F.col(
                "successful_settlement_amount"
            )
        )
        > F.lit(0.01)
    )
    .count()
)

status_attempt_counts_df = (
    payments_source_df
    .groupBy(
        "payment_status",
        "attempt_number",
    )
    .count()
    .orderBy(
        "payment_status",
        "attempt_number",
    )
)

actual_status_attempt_counts = {
    (
        row["payment_status"],
        int(row["attempt_number"]),
    ): int(row["count"])
    for row in status_attempt_counts_df.collect()
}

actual_status_counts = {}

for (
    payment_status,
    attempt_number,
), attempt_count in (
    actual_status_attempt_counts.items()
):
    actual_status_counts[payment_status] = (
        actual_status_counts.get(
            payment_status,
            0,
        )
        + attempt_count
    )

operation_counts_df = (
    payments_source_df
    .groupBy("operation")
    .count()
)

actual_operation_counts = {
    row["operation"]: int(row["count"])
    for row in operation_counts_df.collect()
}

assert (
    payment_event_count
    == EXPECTED_PAYMENT_EVENT_COUNT
)

assert duplicate_payment_count == 0
assert duplicate_provider_transaction_count == 0
assert duplicate_attempt_count == 0

assert (
    actual_status_attempt_counts
    == EXPECTED_STATUS_ATTEMPT_COUNTS
)

assert (
    actual_status_counts
    == EXPECTED_PAYMENT_STATUS_COUNTS
)

assert (
    actual_operation_counts
    == EXPECTED_OPERATION_COUNTS
)

validation_error_columns = [
    "null_required_field_count",
    "orphan_invoice_count",
    "ownership_mismatch_count",
    "invalid_transaction_count",
    "invalid_domain_count",
    "invalid_payment_date_count",
    "invalid_settlement_state_count",
    "invalid_failure_state_count",
    "invalid_invoice_status_mapping_count",
]

for error_column in validation_error_columns:
    error_count = int(
        payment_source_metrics[error_column]
    )

    assert error_count == 0, (
        f"{error_column}: {error_count}"
    )

assert invalid_attempt_sequence_count == 0
assert invalid_retry_predecessor_count == 0
assert paid_invoices_without_success_count == 0
assert past_due_invoices_without_failure_count == 0
assert duplicate_successful_invoice_count == 0
assert voided_invoice_payment_count == 0
assert paid_settlement_mismatch_count == 0

transaction_total = Decimal(
    str(
        payment_source_metrics[
            "transaction_total"
        ]
    )
)

settlement_total = Decimal(
    str(
        payment_source_metrics[
            "settlement_total"
        ]
    )
)

assert (
    abs(
        transaction_total
        - EXPECTED_TRANSACTION_TOTAL
    )
    <= Decimal("0.01")
)

assert (
    abs(
        settlement_total
        - EXPECTED_SETTLEMENT_TOTAL
    )
    <= Decimal("0.01")
)

print(
    f"Validated payment rows: "
    f"{payment_event_count:,}"
)

print(
    f"Distinct payment IDs: "
    f"{distinct_payment_id_count:,}"
)

print(
    "Distinct provider transaction IDs: "
    f"{distinct_provider_transaction_count:,}"
)

print(
    f"Null required fields: "
    f"{payment_source_metrics['null_required_field_count']:,}"
)

print(
    f"Duplicate attempts: "
    f"{duplicate_attempt_count:,}"
)

print(
    f"Orphan invoices: "
    f"{payment_source_metrics['orphan_invoice_count']:,}"
)

print(
    f"Ownership mismatches: "
    f"{payment_source_metrics['ownership_mismatch_count']:,}"
)

print(
    f"Invalid transactions: "
    f"{payment_source_metrics['invalid_transaction_count']:,}"
)

print(
    f"Invalid payment states: "
    f"{payment_source_metrics['invalid_settlement_state_count']:,}"
)

print(
    f"Invalid failure states: "
    f"{payment_source_metrics['invalid_failure_state_count']:,}"
)

print(
    f"Invalid payment dates: "
    f"{payment_source_metrics['invalid_payment_date_count']:,}"
)

print(
    f"Invalid attempt sequences: "
    f"{invalid_attempt_sequence_count:,}"
)

print(
    f"Invalid retry predecessors: "
    f"{invalid_retry_predecessor_count:,}"
)

print(
    "Invalid invoice/status mappings: "
    f"{payment_source_metrics['invalid_invoice_status_mapping_count']:,}"
)

print(
    "Paid invoices without success: "
    f"{paid_invoices_without_success_count:,}"
)

print(
    "Past Due invoices without failure: "
    f"{past_due_invoices_without_failure_count:,}"
)

print(
    "Duplicate successful invoices: "
    f"{duplicate_successful_invoice_count:,}"
)

print(
    f"Voided invoice payments: "
    f"{voided_invoice_payment_count:,}"
)

print(
    f"Paid settlement mismatches: "
    f"{paid_settlement_mismatch_count:,}"
)

print(
    f"Transaction total: "
    f"{transaction_total:,.2f}"
)

print(
    f"Settlement total: "
    f"{settlement_total:,.2f}"
)

display(
    status_attempt_counts_df
)

## 3. Ingest Payment Events into Bronze

Use Databricks Auto Loader to incrementally ingest payment-attempt JSON files into a Delta Bronze table. Add operational lineage, rescued-data support, deterministic record hashes, and a dedicated checkpoint.

In [0]:
payment_record_hash_columns = [
    F.coalesce(
        F.col(column_name).cast("string"),
        F.lit("<NULL>"),
    )
    for column_name in PAYMENT_SOURCE_COLUMNS
]

payment_events_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        SOURCE_FORMAT,
    )
    .option(
        "cloudFiles.schemaLocation",
        PAYMENTS_SCHEMA_PATH,
    )
    .option(
        "cloudFiles.schemaEvolutionMode",
        "rescue",
    )
    .option(
        "rescuedDataColumn",
        "_rescued_data",
    )
    .schema(PAYMENT_EVENT_SCHEMA)
    .load(PAYMENTS_SOURCE_PATH)
    .withColumn(
        "_source_system",
        F.lit(SOURCE_SYSTEM),
    )
    .withColumn(
        "_source_entity",
        F.lit(SOURCE_ENTITY),
    )
    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path"),
    )
    .withColumn(
        "_source_file_name",
        F.col("_metadata.file_name"),
    )
    .withColumn(
        "_source_file_size",
        F.col("_metadata.file_size"),
    )
    .withColumn(
        "_source_file_modification_time",
        F.col("_metadata.file_modification_time"),
    )
    .withColumn(
        "_ingested_at",
        F.current_timestamp(),
    )
    .withColumn(
        "_ingestion_date",
        F.to_date("_ingested_at"),
    )
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *payment_record_hash_columns,
            ),
            256,
        ),
    )
    .select(
        *PAYMENT_BRONZE_COLUMNS
    )
)

payment_bronze_query = (
    payment_events_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        PAYMENTS_CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        PAYMENTS_BRONZE_TABLE
    )
)

payment_bronze_query.awaitTermination()

print(
    "Payment Auto Loader ingestion completed."
)

print(
    f"Bronze table: "
    f"{PAYMENTS_BRONZE_TABLE}"
)

print(
    f"Canonical source: "
    f"{PAYMENTS_SOURCE_PATH}"
)

print(
    f"Checkpoint: "
    f"{PAYMENTS_CHECKPOINT_PATH}"
)

## 4. Validate and Reconcile the Bronze Payment Table

Validate Bronze payment volume, identifier uniqueness, required source and metadata fields, rescued data, canonical lineage, payment-state distribution, financial totals, Delta format, and exact source-to-Bronze content reconciliation.

In [0]:
bronze_payments_df = spark.table(
    PAYMENTS_BRONZE_TABLE
)

required_metadata_columns = [
    "_source_system",
    "_source_entity",
    "_source_file_path",
    "_source_file_name",
    "_source_file_size",
    "_source_file_modification_time",
    "_ingested_at",
    "_ingestion_date",
    "_record_hash",
]

required_metadata_is_missing = None

for column_name in required_metadata_columns:
    missing_condition = (
        F.col(column_name).isNull()
        | (
            F.trim(
                F.col(column_name).cast("string")
            )
            == F.lit("")
        )
    )

    required_metadata_is_missing = (
        missing_condition
        if required_metadata_is_missing is None
        else required_metadata_is_missing
        | missing_condition
    )

invalid_lineage_condition = (
    (F.col("_source_system") != SOURCE_SYSTEM)
    | (F.col("_source_entity") != SOURCE_ENTITY)
    | (
        ~F.col("_source_file_path").contains(
            PAYMENTS_SOURCE_PATH
        )
    )
)

bronze_payment_metrics = (
    bronze_payments_df
    .agg(
        F.count("*").alias(
            "bronze_payment_event_count"
        ),

        F.countDistinct(
            "payment_id"
        ).alias(
            "distinct_payment_id_count"
        ),

        F.countDistinct(
            "provider_transaction_id"
        ).alias(
            "distinct_provider_transaction_count"
        ),

        F.countDistinct(
            F.struct(
                "invoice_id",
                "attempt_number",
            )
        ).alias(
            "distinct_invoice_attempt_count"
        ),

        F.countDistinct(
            "_record_hash"
        ).alias(
            "distinct_record_hash_count"
        ),

        F.sum(
            F.when(
                required_field_is_missing,
                1,
            ).otherwise(0)
        ).alias(
            "null_required_source_field_count"
        ),

        F.sum(
            F.when(
                required_metadata_is_missing,
                1,
            ).otherwise(0)
        ).alias(
            "null_required_metadata_field_count"
        ),

        F.sum(
            F.when(
                F.col("_rescued_data").isNotNull()
                & (
                    F.trim(F.col("_rescued_data"))
                    != F.lit("")
                ),
                1,
            ).otherwise(0)
        ).alias(
            "rescued_data_row_count"
        ),

        F.sum(
            F.when(
                invalid_lineage_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_lineage_row_count"
        ),

        F.sum(
            F.when(
                F.col("_source_file_path").contains(
                    "/initial_load/"
                ),
                1,
            ).otherwise(0)
        ).alias(
            "initial_load_event_count"
        ),

        F.round(
            F.sum("transaction_amount"),
            2,
        ).alias(
            "transaction_total"
        ),

        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settlement_total"
        ),
    )
    .first()
    .asDict()
)

source_reconciliation_df = (
    payments_source_df
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(
                        F.col(column_name).cast("string"),
                        F.lit("<NULL>"),
                    )
                    for column_name
                    in PAYMENT_SOURCE_COLUMNS
                ],
            ),
            256,
        ),
    )
    .select(
        "payment_id",
        "provider_transaction_id",
        "_record_hash",
    )
)

bronze_reconciliation_df = (
    bronze_payments_df
    .select(
        "payment_id",
        "provider_transaction_id",
        "_record_hash",
    )
)

source_missing_from_bronze_count = (
    source_reconciliation_df
    .exceptAll(bronze_reconciliation_df)
    .count()
)

unexpected_bronze_record_count = (
    bronze_reconciliation_df
    .exceptAll(source_reconciliation_df)
    .count()
)

source_bronze_mismatch_count = (
    source_missing_from_bronze_count
    + unexpected_bronze_record_count
)

bronze_status_attempt_counts_df = (
    bronze_payments_df
    .groupBy(
        "payment_status",
        "attempt_number",
    )
    .count()
    .orderBy(
        "payment_status",
        "attempt_number",
    )
)

actual_bronze_status_attempt_counts = {
    (
        row["payment_status"],
        int(row["attempt_number"]),
    ): int(row["count"])
    for row in (
        bronze_status_attempt_counts_df.collect()
    )
}

bronze_operation_counts_df = (
    bronze_payments_df
    .groupBy("operation")
    .count()
)

actual_bronze_operation_counts = {
    row["operation"]: int(row["count"])
    for row in bronze_operation_counts_df.collect()
}

target_detail = (
    spark.sql(
        f"DESCRIBE DETAIL {PAYMENTS_BRONZE_TABLE}"
    )
    .select("format")
    .first()
)

target_format = target_detail["format"].lower()

bronze_payment_event_count = int(
    bronze_payment_metrics[
        "bronze_payment_event_count"
    ]
)

distinct_payment_id_count = int(
    bronze_payment_metrics[
        "distinct_payment_id_count"
    ]
)

distinct_provider_transaction_count = int(
    bronze_payment_metrics[
        "distinct_provider_transaction_count"
    ]
)

distinct_invoice_attempt_count = int(
    bronze_payment_metrics[
        "distinct_invoice_attempt_count"
    ]
)

distinct_record_hash_count = int(
    bronze_payment_metrics[
        "distinct_record_hash_count"
    ]
)

bronze_transaction_total = Decimal(
    str(
        bronze_payment_metrics[
            "transaction_total"
        ]
    )
)

bronze_settlement_total = Decimal(
    str(
        bronze_payment_metrics[
            "settlement_total"
        ]
    )
)

assert (
    bronze_payment_event_count
    == EXPECTED_PAYMENT_EVENT_COUNT
), (
    "Unexpected Bronze payment count: "
    f"{bronze_payment_event_count}"
)

assert (
    distinct_payment_id_count
    == EXPECTED_PAYMENT_EVENT_COUNT
), "Duplicate Bronze payment IDs found."

assert (
    distinct_provider_transaction_count
    == EXPECTED_PAYMENT_EVENT_COUNT
), "Duplicate provider transaction IDs found."

assert (
    distinct_invoice_attempt_count
    == EXPECTED_PAYMENT_EVENT_COUNT
), "Duplicate invoice attempt numbers found."

assert (
    distinct_record_hash_count
    == EXPECTED_PAYMENT_EVENT_COUNT
), "Duplicate Bronze record hashes found."

assert (
    int(
        bronze_payment_metrics[
            "null_required_source_field_count"
        ]
    )
    == 0
), "Null required source fields found."

assert (
    int(
        bronze_payment_metrics[
            "null_required_metadata_field_count"
        ]
    )
    == 0
), "Null required metadata fields found."

assert (
    int(
        bronze_payment_metrics[
            "rescued_data_row_count"
        ]
    )
    == 0
), "Rescued-data rows found."

assert (
    int(
        bronze_payment_metrics[
            "invalid_lineage_row_count"
        ]
    )
    == 0
), "Invalid canonical lineage rows found."

assert (
    int(
        bronze_payment_metrics[
            "initial_load_event_count"
        ]
    )
    == EXPECTED_PAYMENT_EVENT_COUNT
), "Unexpected initial-load event count."

assert (
    actual_bronze_status_attempt_counts
    == EXPECTED_STATUS_ATTEMPT_COUNTS
), (
    "Unexpected Bronze status/attempt counts: "
    f"{actual_bronze_status_attempt_counts}"
)

assert (
    actual_bronze_operation_counts
    == EXPECTED_OPERATION_COUNTS
), (
    "Unexpected Bronze operation counts: "
    f"{actual_bronze_operation_counts}"
)

assert source_bronze_mismatch_count == 0, (
    "Source/Bronze content mismatches: "
    f"{source_bronze_mismatch_count}"
)

assert (
    abs(
        bronze_transaction_total
        - EXPECTED_TRANSACTION_TOTAL
    )
    <= Decimal("0.01")
), (
    "Unexpected Bronze transaction total: "
    f"{bronze_transaction_total}"
)

assert (
    abs(
        bronze_settlement_total
        - EXPECTED_SETTLEMENT_TOTAL
    )
    <= Decimal("0.01")
), (
    "Unexpected Bronze settlement total: "
    f"{bronze_settlement_total}"
)

assert target_format == "delta", (
    f"Unexpected target format: {target_format}"
)

print(
    f"Bronze payment events: "
    f"{bronze_payment_event_count:,}"
)

print(
    f"Distinct Bronze payment IDs: "
    f"{distinct_payment_id_count:,}"
)

print(
    "Distinct provider transaction IDs: "
    f"{distinct_provider_transaction_count:,}"
)

print(
    "Distinct invoice attempts: "
    f"{distinct_invoice_attempt_count:,}"
)

print(
    f"Distinct record hashes: "
    f"{distinct_record_hash_count:,}"
)

print(
    "Null required source fields: "
    f"{bronze_payment_metrics['null_required_source_field_count']:,}"
)

print(
    "Null required metadata fields: "
    f"{bronze_payment_metrics['null_required_metadata_field_count']:,}"
)

print(
    "Rescued-data rows: "
    f"{bronze_payment_metrics['rescued_data_row_count']:,}"
)

print(
    "Invalid canonical lineage rows: "
    f"{bronze_payment_metrics['invalid_lineage_row_count']:,}"
)

print(
    "Initial-load events: "
    f"{bronze_payment_metrics['initial_load_event_count']:,}"
)

print(
    f"Source/Bronze content mismatches: "
    f"{source_bronze_mismatch_count:,}"
)

print(
    f"Bronze transaction total: "
    f"{bronze_transaction_total:,.2f}"
)

print(
    f"Bronze settlement total: "
    f"{bronze_settlement_total:,.2f}"
)

print(
    f"Target format: {target_format}"
)

display(
    bronze_status_attempt_counts_df
)

## 5. Validate Auto Loader Idempotency

Rerun the payment ingestion using the existing Auto Loader checkpoint and confirm that previously processed payment files are not ingested again.

In [0]:
rows_before_idempotency_rerun = (
    spark.table(PAYMENTS_BRONZE_TABLE)
    .count()
)

payment_idempotency_query = (
    payment_events_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        PAYMENTS_CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        PAYMENTS_BRONZE_TABLE
    )
)

payment_idempotency_query.awaitTermination()

rows_after_idempotency_rerun = (
    spark.table(PAYMENTS_BRONZE_TABLE)
    .count()
)

rows_added_during_rerun = (
    rows_after_idempotency_rerun
    - rows_before_idempotency_rerun
)

assert (
    rows_before_idempotency_rerun
    == EXPECTED_PAYMENT_EVENT_COUNT
), (
    "Unexpected row count before rerun: "
    f"{rows_before_idempotency_rerun}"
)

assert (
    rows_after_idempotency_rerun
    == EXPECTED_PAYMENT_EVENT_COUNT
), (
    "Unexpected row count after rerun: "
    f"{rows_after_idempotency_rerun}"
)

assert rows_added_during_rerun == 0, (
    "Auto Loader idempotency failed. "
    f"Rows added during rerun: "
    f"{rows_added_during_rerun}"
)

print(
    "Rows before idempotency rerun: "
    f"{rows_before_idempotency_rerun:,}"
)

print(
    "Rows after idempotency rerun: "
    f"{rows_after_idempotency_rerun:,}"
)

print(
    "Rows added during rerun: "
    f"{rows_added_during_rerun:,}"
)

print(
    "Payment Bronze ingestion is idempotent."
)